In [30]:
#Imports
import torch
import torch.nn as nn
import os
import matplotlib.pyplot as plt

In [22]:
#Generator (U-Net)
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, down=True, use_dropout=False):
        super().__init__()
        
        if down:
            # Downsampling block (encoder)
            self.block = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=4, 
                         stride=2, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.LeakyReLU(0.2, inplace=True)
            )
        else:
            # Upsampling block (decoder)
            self.block = nn.Sequential(
                nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4,
                                  stride=2, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            )
        
        self.use_dropout = use_dropout
        self.dropout     = nn.Dropout(0.5)

    def forward(self, x):
        x = self.block(x)
        if self.use_dropout:
            x = self.dropout(x)
        return x


class Generator(nn.Module):
    """
    U-Net Generator
    Input:  L channel  (1, 256, 256)
    Output: AB channels (2, 256, 256)
    """
    def __init__(self):
        super().__init__()

        # ---- Encoder (downsampling) ----
        # No batchnorm on first layer
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )                                           # (64, 128, 128)
        self.enc2 = UNetBlock(64,   128)            # (128, 64, 64)
        self.enc3 = UNetBlock(128,  256)            # (256, 32, 32)
        self.enc4 = UNetBlock(256,  512)            # (512, 16, 16)
        self.enc5 = UNetBlock(512,  512)            # (512, 8, 8)
        self.enc6 = UNetBlock(512,  512)            # (512, 4, 4)
        self.enc7 = UNetBlock(512,  512)            # (512, 2, 2)

        # ---- Bottleneck ----
        self.bottleneck = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )                                           # (512, 1, 1)

        # ---- Decoder (upsampling) with skip connections ----
        # Input channels doubled because of skip connections
        self.dec1 = UNetBlock(512,  512, down=False, use_dropout=True)   # (512, 2, 2)
        self.dec2 = UNetBlock(1024, 512, down=False, use_dropout=True)   # (512, 4, 4)
        self.dec3 = UNetBlock(1024, 512, down=False, use_dropout=True)   # (512, 8, 8)
        self.dec4 = UNetBlock(1024, 512, down=False)                     # (512, 16, 16)
        self.dec5 = UNetBlock(1024, 256, down=False)                     # (256, 32, 32)
        self.dec6 = UNetBlock(512,  128, down=False)                     # (128, 64, 64)
        self.dec7 = UNetBlock(256,   64, down=False)                     # (64, 128, 128)

        # Final layer — output AB channels
        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, 2, kernel_size=4, stride=2, padding=1),
            nn.Tanh()                               # output range (-1, 1)
        )                                           # (2, 256, 256)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)
        e6 = self.enc6(e5)
        e7 = self.enc7(e6)

        # Bottleneck
        b = self.bottleneck(e7)

        # Decoder with skip connections (concatenate encoder features)
        d1 = self.dec1(b)
        d2 = self.dec2(torch.cat([d1, e7], dim=1))
        d3 = self.dec3(torch.cat([d2, e6], dim=1))
        d4 = self.dec4(torch.cat([d3, e5], dim=1))
        d5 = self.dec5(torch.cat([d4, e4], dim=1))
        d6 = self.dec6(torch.cat([d5, e3], dim=1))
        d7 = self.dec7(torch.cat([d6, e2], dim=1))

        return self.final(torch.cat([d7, e1], dim=1))

In [23]:
#Discriminator (PatchGAN)
class Discriminator(nn.Module):
    """
    PatchGAN Discriminator
    Input:  L + AB concatenated (3, 256, 256)
    Output: Patch predictions   (1, 30, 30)
    Each value = real/fake score for a 70x70 patch
    """
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            # Layer 1 — no batchnorm on first layer
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 2
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 3
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 4 — stride 1
            nn.Conv2d(256, 512, kernel_size=4, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            # Output layer — stride 1, single channel
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1)
            # No sigmoid — handled by loss function
        )

    def forward(self, L, AB):
        # Concatenate L and AB along channel dimension
        x = torch.cat([L, AB], dim=1)   # (3, 256, 256)
        return self.model(x)

In [24]:
#Weight Initialization
def init_weights(model):
    """
    Initialize weights using normal distribution
    as described in the original Pix2Pix paper
    """
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight.data, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias.data, 0)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.normal_(m.weight.data, mean=1.0, std=0.02)
            nn.init.constant_(m.bias.data, 0)

    print(f" Weights initialized for {model.__class__.__name__}")
    return model

In [25]:
#Test model shapes (sanity check)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize models
G = init_weights(Generator()).to(device)
D = init_weights(Discriminator()).to(device)

# Test with dummy input
L_dummy  = torch.randn(1, 1, 256, 256).to(device)   # fake L channel
AB_dummy = torch.randn(1, 2, 256, 256).to(device)   # fake AB channels

# Test Generator
AB_pred = G(L_dummy)
print(f"\nGenerator:")
print(f"  Input shape:  {L_dummy.shape}")
print(f"  Output shape: {AB_pred.shape}")    # should be (1, 2, 256, 256)

# Test Discriminator
D_pred = D(L_dummy, AB_pred)
print(f"\nDiscriminator:")
print(f"  Input shape:  {torch.cat([L_dummy, AB_pred], dim=1).shape}")
print(f"  Output shape: {D_pred.shape}")     # should be (1, 1, 30, 30)

# Count parameters
G_params = sum(p.numel() for p in G.parameters())
D_params = sum(p.numel() for p in D.parameters())
print(f"\nGenerator parameters:     {G_params:,}")
print(f"Discriminator parameters: {D_params:,}")

Using device: cuda
 Weights initialized for Generator
 Weights initialized for Discriminator

Generator:
  Input shape:  torch.Size([1, 1, 256, 256])
  Output shape: torch.Size([1, 2, 256, 256])

Discriminator:
  Input shape:  torch.Size([1, 3, 256, 256])
  Output shape: torch.Size([1, 1, 30, 30])

Generator parameters:     54,410,434
Discriminator parameters: 2,765,633


Below are to generate images/graphs/tabnles for the report(as my first one was't detailed enough....)

In [26]:
#layer summary table
def print_model_summary(G, D):
    def count_params(model):
        total     = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        return total, trainable

    g_total, g_train = count_params(G)
    d_total, d_train = count_params(D)

    print(f"\n{'='*65}")
    print(f"  {'MODEL SUMMARY TABLE':^61}")
    print(f"{'='*65}")
    print(f"  {'Component':<20} {'Type':<12} {'Parameters':>12} {'Size (MB)':>10}")
    print(f"  {'-'*61}")
    print(f"  {'Generator':<20} {'U-Net':<12} {g_total:>12,} {g_total*4/1024**2:>9.1f}MB")
    print(f"  {'Discriminator':<20} {'PatchGAN':<12} {d_total:>12,} {d_total*4/1024**2:>9.1f}MB")
    print(f"  {'-'*61}")
    print(f"  {'Total':<20} {'':<12} {g_total+d_total:>12,} {(g_total+d_total)*4/1024**2:>9.1f}MB")
    print(f"{'='*65}")

    # Generator layer breakdown
    print(f"\n{'='*65}")
    print(f"  {'GENERATOR LAYER BREAKDOWN':^61}")
    print(f"{'='*65}")
    print(f"  {'Layer':<12} {'Type':<20} {'Input Shape':<20} {'Output Shape':<15}")
    print(f"  {'-'*61}")

    layers = [
        ("enc1",       "Conv2d + LeakyReLU",   "(1, 256, 256)",   "(64, 128, 128)"),
        ("enc2",       "UNetBlock (down)",      "(64, 128, 128)",  "(128, 64, 64)" ),
        ("enc3",       "UNetBlock (down)",      "(128, 64, 64)",   "(256, 32, 32)" ),
        ("enc4",       "UNetBlock (down)",      "(256, 32, 32)",   "(512, 16, 16)" ),
        ("enc5",       "UNetBlock (down)",      "(512, 16, 16)",   "(512, 8, 8)"   ),
        ("enc6",       "UNetBlock (down)",      "(512, 8, 8)",     "(512, 4, 4)"   ),
        ("enc7",       "UNetBlock (down)",      "(512, 4, 4)",     "(512, 2, 2)"   ),
        ("bottleneck", "Conv2d + ReLU",         "(512, 2, 2)",     "(512, 1, 1)"   ),
        ("dec1",       "UNetBlock (up+skip)",   "(512, 1, 1)",     "(512, 2, 2)"   ),
        ("dec2",       "UNetBlock (up+skip)",   "(1024, 2, 2)",    "(512, 4, 4)"   ),
        ("dec3",       "UNetBlock (up+skip)",   "(1024, 4, 4)",    "(512, 8, 8)"   ),
        ("dec4",       "UNetBlock (up+skip)",   "(1024, 8, 8)",    "(512, 16, 16)" ),
        ("dec5",       "UNetBlock (up+skip)",   "(1024, 16, 16)",  "(256, 32, 32)" ),
        ("dec6",       "UNetBlock (up+skip)",   "(512, 32, 32)",   "(128, 64, 64)" ),
        ("dec7",       "UNetBlock (up+skip)",   "(256, 64, 64)",   "(64, 128, 128)"),
        ("final",      "ConvTranspose2d+Tanh",  "(128, 128, 128)", "(2, 256, 256)" ),
    ]

    for name, layer_type, inp, out in layers:
        print(f"  {name:<12} {layer_type:<20} {inp:<20} {out:<15}")

    print(f"{'='*65}")

print_model_summary(G, D)


                       MODEL SUMMARY TABLE                     
  Component            Type           Parameters  Size (MB)
  -------------------------------------------------------------
  Generator            U-Net          54,410,434     207.6MB
  Discriminator        PatchGAN        2,765,633      10.6MB
  -------------------------------------------------------------
  Total                               57,176,067     218.1MB

                    GENERATOR LAYER BREAKDOWN                  
  Layer        Type                 Input Shape          Output Shape   
  -------------------------------------------------------------
  enc1         Conv2d + LeakyReLU   (1, 256, 256)        (64, 128, 128) 
  enc2         UNetBlock (down)     (64, 128, 128)       (128, 64, 64)  
  enc3         UNetBlock (down)     (128, 64, 64)        (256, 32, 32)  
  enc4         UNetBlock (down)     (256, 32, 32)        (512, 16, 16)  
  enc5         UNetBlock (down)     (512, 16, 16)        (512, 8, 8)  

In [31]:
#parameter distribution graph
def plot_parameter_distribution(G):
    # Count parameters per section
    sections = {
        "Encoder":    ["enc1", "enc2", "enc3", "enc4", "enc5", "enc6", "enc7"],
        "Bottleneck": ["bottleneck"],
        "Decoder":    ["dec1", "dec2", "dec3", "dec4", "dec5", "dec6", "dec7"],
        "Final":      ["final"]
    }

    section_params = {}
    for section, layer_names in sections.items():
        total = 0
        for name in layer_names:
            layer = getattr(G, name)
            total += sum(p.numel() for p in layer.parameters())
        section_params[section] = total

    colors  = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
    labels  = [f"{k}\n{v/1e6:.1f}M params" for k, v in section_params.items()]
    sizes   = list(section_params.values())
    explode = (0.05, 0.05, 0.05, 0.05)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Generator Parameter Distribution", fontsize=14, fontweight="bold")

    # Pie chart
    wedges, texts, autotexts = axes[0].pie(
        sizes,
        labels    = labels,
        colors    = colors,
        explode   = explode,
        autopct   = "%1.1f%%",
        startangle= 90,
        textprops = {"fontsize": 10}
    )
    axes[0].set_title("Parameter Split by Section")

    # Bar chart
    axes[1].bar(
        list(section_params.keys()),
        [v/1e6 for v in section_params.values()],
        color     = colors,
        edgecolor = "white",
        linewidth = 0.5
    )
    axes[1].set_title("Parameters per Section (Millions)")
    axes[1].set_xlabel("Section")
    axes[1].set_ylabel("Parameters (M)")
    for i, (k, v) in enumerate(section_params.items()):
        axes[1].text(i, v/1e6 + 0.3, f"{v/1e6:.1f}M", 
                    ha="center", va="bottom", fontsize=10)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, "parameter_distribution.png"), dpi=150, bbox_inches="tight")
    plt.close()
    print("Parameter distribution saved to outputs/parameter_distribution.png")

# Need OUTPUT_PATH defined here too
OUTPUT_PATH = os.path.expanduser("~/colorizer_gan/outputs")
os.makedirs(OUTPUT_PATH, exist_ok=True)

plot_parameter_distribution(G)

Parameter distribution saved to outputs/parameter_distribution.png
